In [16]:
from sparse_ops import initialize_nonnegative_tucker, unfold_from_vectorized_sparse, ptl_tucker_to_tensor
from tucker_tensor import SparseTupleTensor
from distance import kl_factor_update_largedim, kl_factor_update
from utils import select_gpu
import pytensorlab as ptl
import tensorly_custom as tl
import cupy as cp
import numpy as np
import cupyx.scipy.sparse as cpx_sparse
tl.set_backend("cupy")
device = select_gpu()
dataset = "fineweb_large"
method = "sc"
dim = 2000
def inspect(element):
    element_var = globals()[element]
    print(f"{element}, {element_var.shape}, {type(element_var)}")
    ptl_el = "ptl_"+element
    try:
        inspect(ptl_el)
    except:
        return

sparse_tensor = SparseTupleTensor.load_from_disk(
                dataset=dataset,
                method=method,
                dims=dim,
                map_location="cpu",
            )
sparse_tensor.tensor_to_sparse("sparse")
ptl_tensor = ptl.SparseTensor(data=sparse_tensor.tensor.data,
                        indices=sparse_tensor.tensor.coords,
                        shape=sparse_tensor.tensor.shape)
sparse_tensor.tensor_to_sparse("torch")
sparse_tensor.tensor_to_sparse("cupy")
sparse_tensor.tensor_to_sparse("cupy")
rank = [150, 150, 150]
tensor = sparse_tensor.tensor
shape = tuple(sparse_tensor.shape)
modes = list(range(len(rank)))
init = "random"
random_state = 1
epsilon = 1e-12
core, factors = initialize_nonnegative_tucker(sparse_tensor, shape, rank, modes, init, random_state)

Selected GPU 2 with 454.62 MB used memory.


In [17]:
ptl_factors = []
for mode in modes:
    # Sparse unfolding for this mode
    X = unfold_from_vectorized_sparse(tensor, shape, mode)
    core_np = tl.to_numpy(core)
    factors_np = [tl.to_numpy(f) for f in factors]
    # Dense reconstruction excluding current factor, unfolded along mode
    ptl_tucker = ptl.TuckerTensor(core=core_np,
                                  factors=factors_np)
    ptl_Z = ptl_tucker_to_tensor(ptl_tucker, skip_factor=mode)
    ptl_Z = ptl.tens2mat(ptl_Z, mode)
    # Z = tucker_to_tensor((core, factors), skip_factor=mode)
    # Z = unfold(Z, mode)  # (R, K) after unfold
    # inspect("Z")
    A = factors[mode]  # (I_mode, R)
    rows = X.row
    cols = X.col
    vals = X.data

    # Compute reconstruction only at nonzeros: R_nz = sum_r A[i,r] * Z[r,j]
    # A_rows: (nnz, R)
    A_rows = A[rows, :]
    # Z_cols_T: (nnz, R) because Z[:, cols] is (R, nnz)
    # Z_cols_T = tl.transpose(Z[:, cols])
    ptl_Z_cols_T = tl.transpose(ptl_Z[:, cols.get()])
    ptl_Z_cols_T = cp.asarray(ptl_Z_cols_T)
    # R_nz = tl.sum(A_rows * Z_cols_T, axis=1)
    # R_nz = tl.clip(R_nz, a_min=epsilon, a_max=None)
    ptl_R_nz = tl.sum(A_rows * ptl_Z_cols_T, axis=1)
    ptl_R_nz = tl.clip(ptl_R_nz, a_min=epsilon, a_max=None)
    # inspect("Z_cols_T")
    # inspect("R_nz")

    # W = X / (A Z) at nonzeros
    # W_data = vals / R_nz
    # W = cpx_sparse.coo_matrix((W_data, (rows, cols)), shape=X.shape)
    ptl_W_data = vals / ptl_R_nz
    ptl_W = cpx_sparse.coo_matrix((ptl_W_data, (rows, cols)), shape=X.shape)
    # inspect("W_data")
    # inspect("W")

    # numerator = W @ Z^T   -> (I_mode, R)
    # numerator = W @ tl.transpose(Z)
    # inspect("W")
    # inspect("Z")
    # ptl_numerator = ptl_W @ tl.transpose(ptl_Z)
    ptl_numerator = ptl_W @ tl.transpose(cp.asarray(ptl_Z))
    # inspect("numerator")

    # denominator = sum_j Z[r,j] broadcast to (I_mode, R)
    # den_row = tl.sum(Z, axis=1)  # (R,)
    # denominator = den_row[np.newaxis, :]
    # denominator = tl.clip(denominator, a_min=epsilon, a_max=None)
    ptl_den_row = tl.sum(ptl_Z, axis=1)
    ptl_denominator = ptl_den_row[np.newaxis, :]
    ptl_denominator = tl.clip(ptl_denominator, a_min=epsilon, a_max=None)
    # inspect("den_row")
    # inspect("denominator")

    # Multiplicative update
    # new_A = A * (numerator / (denominator + 1e-12))
    # new_A = tl.clip(new_A, a_min=epsilon, a_max=None)
    A = A * (ptl_numerator / (cp.asarray(ptl_denominator + 1e-12)))
    ptl_factors.append(A)


In [19]:
func_factors = []
for mode in modes:
    func_factors.append(kl_factor_update_largedim(tensor, shape, core, factors, mode, epsilon))
old_func_factors = []
for mode in modes:
    old_func_factors.append(kl_factor_update(tensor, shape, core, factors, mode, epsilon))

In [23]:
for f1, f2, f3 in zip(func_factors, old_func_factors, old_func_factors):
    print(cp.allclose(f1, f2, rtol=epsilon))
    print(cp.allclose(f2, f3, rtol=epsilon))
    print(cp.allclose(f3, f1, rtol=epsilon))


True
True
True
True
True
True
True
True
True
